In [ ]:
import kagglehub
import pandas as pd
import seaborn as sbn
import matplotlib.pyplot as plt
import os

# Download latest version
path = kagglehub.dataset_download("anikannal/solar-power-generation-data")


In [ ]:

gen_df = pd.read_csv(path + '/Plant_1_Generation_Data.csv')
weather_df = pd.read_csv(path+"/Plant_1_Weather_Sensor_Data.csv")

print("Pierwsze 5 wierszy tabeli generacji:")
display(gen_df.head())

print("\nInformacje o kolumnach:")
gen_df.info()

In [ ]:
gen_df['DATE_TIME'] = pd.to_datetime(gen_df['DATE_TIME'])
weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'])

gen_df.info()
weather_df.info()


In [ ]:
df = pd.merge(gen_df, weather_df, on='DATE_TIME', how='left')
df.head()

In [ ]:
df[(df['IRRADIATION'] > 0.1) & (df['DC_POWER'] == 0)].head()

In [ ]:
df = df[df['IRRADIATION'] > 0]

In [ ]:
df['EFFICIENCY'] = df['AC_POWER']/df['IRRADIATION']
df['EFFICIENCY'].describe()

In [ ]:
plt.figure(figsize=(10,6))
sbn.scatterplot(data=df, x='MODULE_TEMPERATURE', y='EFFICIENCY', hue='IRRADIATION', palette='viridis', alpha=0.5)
plt.title('Wpływ temperatury na sprawność')
plt.grid(True)
plt.show

In [ ]:
df = df[df['IRRADIATION'] > 0.8]
plt.figure(figsize=(10,6))
sbn.scatterplot(data=df, x='MODULE_TEMPERATURE', y='EFFICIENCY', hue='IRRADIATION', palette='viridis', alpha=0.5)
plt.title('Wpływ temperatury na sprawność')
plt.grid(True)
plt.show


In [ ]:
columns = ['EFFICIENCY', 'MODULE_TEMPERATURE', 'IRRADIATION']
print(df[columns].corr())

In [ ]:
sbn.regplot(data=df, x='MODULE_TEMPERATURE', y='EFFICIENCY')

In [ ]:
df = df.rename(columns={'SOURCE_KEY_x': 'INVERTER_ID'})
plt.figure(figsize=(15, 8))
sbn.boxplot(data=df, x='INVERTER_ID',y='DC_POWER')
plt.xticks(rotation=90)
plt.show()

In [ ]:
total_yield = df.groupby('INVERTER_ID')['DC_POWER'].sum()

In [ ]:
total_yield.median()

In [ ]:
df.groupby('INVERTER_ID')['DC_POWER'].sum().sort_values()

In [ ]:
x = (2550199.0387204997 - 2.263525e+06) / 2550199.0387204997
print(x)

In [ ]:
faulty_inv = df[df['INVERTER_ID'] == '1BY6WEcLGh8j5v7']
healthy_inv = df[df['INVERTER_ID'] == 'McdE0feGgRqW7Ca']

In [ ]:
plt.plot('DATE_TIME', 'DC_POWER', data=faulty_inv, label='Uszkodzony')
plt.plot('DATE_TIME', 'DC_POWER', data=healthy_inv, label='Wzorzec')
plt.legend()